In [ ]:
!pip install pycuda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 22.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.8/98.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 11.1 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp312-cp312-linux_x86_64.whl size=659447 sha256=ca84c2e19fdc103ee3ac3de551f1d8a970119e3223f974e3c47b2118f6df4fe7
  Stored in directory: /root/.cache/pip/wheels/90/2a/71/75ec0cc316cc0ff494bfffa2935e02580129cb7f859a0cfd8f
Successfully built pycuda


In [8]:
import numpy as np
import pycuda.driver as cuda
import pycuda.autoinit
from pycuda.compiler import SourceModule
import time

# CUDA code for summing vector elements
vector_sum_kernel = """
__global__ void addKernel(int* result, int* a, unsigned int size) {
    int index = blockIdx.x * blockDim.x + threadIdx.x;

    __shared__ int sharedSum[256];

    int localSum = 0;

    if (index < size) {
        localSum = a[index];
    }

    sharedSum[threadIdx.x] = localSum;
    __syncthreads();

    for (int stride = blockDim.x / 2; stride > 0; stride >>= 1) {
        if (threadIdx.x < stride) {
            sharedSum[threadIdx.x] += sharedSum[threadIdx.x + stride];
        }
        __syncthreads();
    }

    if (threadIdx.x == 0) {
        atomicAdd(result, sharedSum[0]);
    }
}
"""

def vector_sum_gpu(vector):
    start_time = time.time()

    vector_gpu = cuda.mem_alloc(vector.nbytes)
    result_gpu = cuda.mem_alloc(np.int32().nbytes)

    initial_value = np.array([0], dtype=np.int32)
    cuda.memcpy_htod(result_gpu, initial_value)

    cuda.memcpy_htod(vector_gpu, vector)

    mod = SourceModule(vector_sum_kernel)
    vector_sum = mod.get_function("addKernel")

    block_size = 256
    grid_size = (len(vector) + block_size - 1) // block_size

    vector_sum(
        result_gpu,
        vector_gpu,
        np.int32(len(vector)),
        block=(block_size, 1, 1),
        grid=(grid_size, 1)
    )

    result = np.empty(1, dtype=np.int32)
    cuda.memcpy_dtoh(result, result_gpu)

    end_time = time.time()

    return result[0], end_time - start_time


def vector_sum_cpu(vector):
    answer = 0
    for elem in vector:
        answer += elem
    return answer


if __name__ == "__main__":
    sizes = [
        2_000,
        4_000,
        8_000,
        16_000,
        32_000,
        64_000,
        128_000,
        256_000,
        512_000,
        1_024_000,
        2_048_000,
        4_096_000,
        8_192_000,
        8_194_000
    ]

    print("Size | CPU time | GPU time |")
    print("----------------------------")

    for vector_size in sizes:
        vector = np.random.randint(1, 10, size=vector_size, dtype=np.int32)

        # CPU
        start_time_cpu = time.time()
        answer_cpu = vector_sum_cpu(vector)
        time_cpu = time.time() - start_time_cpu

        # GPU
        answer_gpu, time_gpu = vector_sum_gpu(vector)

        print(f"{vector_size} | {time_cpu:.6f} | {time_gpu:.6f}")

Size | CPU time | GPU time |
----------------------------
2000 | 0.000244 | 0.002450
4000 | 0.000612 | 0.000880
8000 | 0.001194 | 0.000421
16000 | 0.001376 | 0.000517
32000 | 0.003048 | 0.000432
64000 | 0.007092 | 0.000508
128000 | 0.012634 | 0.000578
256000 | 0.025279 | 0.000694
512000 | 0.048250 | 0.001050
1024000 | 0.103530 | 0.002059
2048000 | 0.204426 | 0.002932
4096000 | 0.411872 | 0.005071
8192000 | 0.804613 | 0.009176
8194000 | 0.805829 | 0.009354
